# Select a GOC-500 model and run real inference

Use the pinned model environment and acquire the documented checksum-bound model assets first. The executed outputs in this notebook are real CPU predictions on the official sample, not historical held-out test metrics. Choose checkpoint IDs below to compare models; FT0 is a smoke checkpoint and is excluded from the experiment comparison.

The optional bounded OPS example evaluates two topologies with eight model calls each on a synthetic uniform hazard. It is distinct from the historical three-scenario GOC OPS experiment. Missing weights are reported as **unavailable**, never as passed numerical results.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display, JSON, Image

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'artifacts/model_catalog.json').is_file(), 'Run from this repository or its notebooks directory.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = ROOT / 'outputs' / ('notebook_' + RUN_ID)
OUT.mkdir(parents=True)


In [2]:
from genco_ops.catalog import models
from genco_ops.workflow import run_workflow
MODEL_IDS = ['goc500.base', 'goc500.m1', 'goc500.m2', 'goc500.m3']
DEMO_MODEL = 'goc500.m2'
RUN_BOUNDED_DEMO = True
available = models(ROOT, 'goc_500')
display(pd.DataFrame([{key: row[key] for key in ('model_id', 'role', 'weights_ready', 'saved_input_ready', 'supported_modes')} for row in available]))

,model_id,role,weights_ready,saved_input_ready,supported_modes
0,goc500.base,base,True,True,"[infer, demo]"
1,goc500.ft0,smoke,True,True,"[infer, demo]"
2,goc500.m1,final,True,True,"[infer, demo]"
3,goc500.m2,final,True,True,"[infer, demo]"
4,goc500.m3,final,True,True,"[infer, demo]"


In [3]:
inference_results = []
for model_id in MODEL_IDS:
    destination = OUT / ('inference_' + model_id)
    plan = run_workflow(ROOT, 'goc_500', model_id, destination, mode='infer')
    if plan['status'] == 'MISSING_ARTIFACTS':
        result = {'model_id': model_id, 'status': 'UNAVAILABLE_MISSING_ASSETS_NOT_A_PASSED_DEMO', 'missing_assets': plan['missing_assets']}
    else:
        result = run_workflow(ROOT, 'goc_500', model_id, destination, mode='infer', execute=True)
    inference_results.append(result)
    display(JSON({'kind': 'saved_case_inference', **result}))
display(pd.DataFrame(inference_results)[['model_id', 'status']])

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

,model_id,status
0,goc500.base,PASS_FINITE_INFERENCE_NOT_AC_FEASIBILITY
1,goc500.m1,PASS_FINITE_INFERENCE_NOT_AC_FEASIBILITY
2,goc500.m2,PASS_FINITE_INFERENCE_NOT_AC_FEASIBILITY
3,goc500.m3,PASS_FINITE_INFERENCE_NOT_AC_FEASIBILITY


A finite model prediction does not certify AC feasibility. The output schemas and normalization differ between model families; the pipeline loads the checkpoint's native adapter and keeps its normalizer unchanged.

Next, apply the selected GOC checkpoint to a bounded fresh topology/service search. The source electrical sample, uniform hazard policy, call budget, and model/input hashes are recorded in its result file. This does not reproduce the absent historical baseline/PAC preparation or independent held-out evaluation population.

In [4]:
if RUN_BOUNDED_DEMO:
    destination = OUT / ('ops_' + DEMO_MODEL)
    plan = run_workflow(ROOT, 'goc_500', DEMO_MODEL, destination, mode='demo')
    if plan['status'] == 'MISSING_ARTIFACTS':
        outcome = {'status': 'UNAVAILABLE_MISSING_ASSETS_NOT_A_PASSED_DEMO', 'missing_assets': plan['missing_assets']}
    else:
        outcome = run_workflow(ROOT, 'goc_500', DEMO_MODEL, destination, mode='demo', execute=True)
        outcome['result'] = str(Path(outcome['result']).relative_to(ROOT))
        record = json.loads((destination / 'demo_result.json').read_text())
        display(pd.DataFrame([{'offline_branch_ids': row['offline_branch_ids'], 'actual_model_calls': row['actual_model_calls'], **{key: row['best'].get(key) for key in ('r_norm', 'l_shed_total', 'j_trade', 'search_objective')}} for row in record['topologies']]))
    display(JSON({'kind': 'bounded_sample_ops', **outcome}))
else:
    display(JSON({'kind': 'bounded_sample_ops', 'status': 'NOT_REQUESTED_NO_NUMERICAL_DEMO_CLAIM'}))

,offline_branch_ids,actual_model_calls,r_norm,l_shed_total,j_trade,search_objective
0,[],8,1.005336,0.016563,0.807581,0.807581
1,[1],8,1.004371,0.016563,0.806810,0.806810


<IPython.core.display.JSON object>

## Compare the corresponding recorded experiment

These observations come from the accepted refined GOC study, not the sample demo above. Rerun notebook 01 for all retained frontier, physics, AC agreement and timing figures.

In [5]:
from genco_ops.results import goc_comparison
display(goc_comparison(ROOT).round(6))

,model,method,finalists,native_j_mean,ac_j_mean,ac_cost_mean,state_nrmse_mean,finite_state_families,absolute_risk_discrepancy_mean,absolute_objective_discrepancy_mean
0,DC,Guided-DC,15,0.260473,0.296249,460634.434452,0.022693,3,0.070265,0.035776
1,M0,Guided-GridSFM (released v1.1),15,0.333630,0.302839,453440.836380,0.207259,7,0.074987,0.030791
2,M1,Guided-GridSFM (FullTop-1000),15,0.308432,0.298451,454311.951927,0.147720,7,0.027569,0.009981
3,M2,Guided-GridSFM (FullTop-1500),15,0.303515,0.299627,451374.564540,0.120588,7,0.017622,0.005112
4,M3,Guided-GridSFM (FullTop+N-1),15,0.309792,0.303622,452511.890414,0.141500,7,0.021816,0.007044
